# Session 2 · Prepare data (on your Mac)

Same script as the Colab version. Apple Silicon has more CPU cores than a basic Colab runtime, and tokenizing is CPU work.

- **Disk:** about 25 GB free (20 GB of shards, plus one 2 GB download at a time).
- **Storage:** `~/chotu-data` (change `STORAGE` below if you want another drive).
- **Resumable:** if the Mac sleeps or you stop it, run cell 4 again and finished files are skipped.
- `caffeinate` keeps the Mac awake while it runs. Keep it plugged in.

In [ ]:
# 1. Paths, cores, disk
import os, shutil, multiprocessing as mp
from pathlib import Path
REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
STORAGE = Path.home() / 'chotu-data'
STORAGE.mkdir(exist_ok=True)
os.chdir(REPO)
n = mp.cpu_count(); free = shutil.disk_usage(STORAGE).free / 1e9
print(f'repo: {REPO}\nstorage: {STORAGE}\nCPU cores: {n} -> full run roughly {28/n:.0f} h\ndisk free: {free:.0f} GB')
assert (REPO / 'scripts' / 'prepare_data.py').exists(), 'Open this notebook from inside the Chotu repo'
assert free > 25, 'Need ~25 GB free disk'

In [ ]:
# 2. Tests (model size + shard format)
!git pull --rebase -q
!python -m pytest -q tests

In [ ]:
# 3. Quick test (~1 min): tokenizer checks + 2 small shards in ~/chotu-data/test/
!python scripts/prepare_data.py --storage "{STORAGE}" --max-docs 2000 --max-files 2

In [ ]:
# 4. Full run (resumable). caffeinate stops the Mac sleeping until it finishes.
!caffeinate -i python scripts/prepare_data.py --storage "{STORAGE}" --local-dir "{STORAGE}/tmp" 

In [ ]:
# 5. Read the report: token count, tokenizer checks, 10 samples
from IPython.display import Markdown
Markdown((STORAGE / 'reports' / 'data_report.md').read_text())

## Gate
- About 10B train tokens, and a validation split exists
- You read the 10 samples: clean English, documents separated, no markup junk

Then add your entry to `docs/PROGRESS.md` and push from your terminal. The shards stay on your Mac; before Session 5 they get uploaded to a private Hugging Face dataset so the H100 machine can fetch them.